# Pipeline Workflow On Small Samples

This notebook runs each pipeline unit independently on a reduced local dataset from `samples/`.

Covered units:
1. Local sample PDF -> text extraction (`pdf_text_extraction` + extraction heuristics)
2. Raw text parsing spot checks (`raw_text_parsing`)
3. `extract_days_from_text_raw`
4. `extract_events_from_days_raw`
5. `filter_midnight_events_from_days_raw`
6. `pair_employee_events_from_days_raw`
7. `turni_enrichment`
8. `turni_employee_summary`

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "samples").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing src/ and samples/")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

SAMPLES_DIR = REPO_ROOT / "samples"
WORK_ROOT = REPO_ROOT / "output" / "notebook_samples"
TEXT_DIR = WORK_ROOT / "text_extracted"
PARSED_DIR = WORK_ROOT / "parsed_from_text"
PAIRS_DIR = WORK_ROOT / "employee_shifts_from_raw"
ENRICHED_DIR = WORK_ROOT / "enriched" / "employee_pairs"
AGG_DIR = WORK_ROOT / "aggregates"
REPORTS_DIR = WORK_ROOT / "reports"

print(f"Repo root: {REPO_ROOT}")
print(f"Samples dir: {SAMPLES_DIR}")
print(f"Work root: {WORK_ROOT}")

In [ ]:
# Tweak these knobs if needed.
MAX_PDFS_PER_FOLDER = 2
RESET_WORKDIR = True

if RESET_WORKDIR and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

for path in [WORK_ROOT, TEXT_DIR, PARSED_DIR, PAIRS_DIR, ENRICHED_DIR, AGG_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Workspace initialized.")

In [ ]:
def run_module(module: str, *args: str, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


def preview_csv(path: Path, n: int = 10) -> None:
    print(f"Preview: {path}")
    df = pd.read_csv(path)
    display(df.head(n))
    print(f"rows={len(df)}, cols={list(df.columns)}")


def first_or_none(paths: list[Path]) -> Path | None:
    return paths[0] if paths else None

## 1) Build a tiny local sample set (PDF -> TXT)

This replaces Drive-specific ingestion so you can test the rest of the workflow locally.

In [ ]:
from src.pdf_text_extraction import extract_text, extract_text_vertical


def quality_score(text: str) -> float:
    total = len(text)
    if total == 0:
        return 0.0
    printable = sum(1 for ch in text if ch.isprintable() or ch in "\n\r\t")
    non_ws = [ch for ch in text if not ch.isspace()]
    alpha = sum(1 for ch in non_ws if ch.isalpha())
    printable_ratio = printable / total
    alpha_ratio = (alpha / len(non_ws)) if non_ws else 0.0
    return round((0.6 * printable_ratio) + (0.4 * min(1.0, alpha_ratio / 0.35)), 6)


def extract_best_local(pdf_path: Path) -> dict[str, object]:
    normal = extract_text(pdf_path)
    vertical = extract_text_vertical(pdf_path)
    normal_score = quality_score(normal)
    vertical_score = quality_score(vertical)

    selected_mode = "normal"
    selected_text = normal
    if vertical.strip() and (not normal.strip() or vertical_score > normal_score + 0.08):
        selected_mode = "vertical"
        selected_text = vertical

    if not selected_text.strip():
        raise RuntimeError(f"Empty text extracted from: {pdf_path}")

    return {
        "text": selected_text,
        "mode": selected_mode,
        "tried_vertical": bool(vertical.strip()),
        "normal_score": normal_score,
        "vertical_score": vertical_score,
    }

sample_folders = [
    ("sample_cartellino", SAMPLES_DIR / "cartellino"),
    ("sample_timbrature_compact", SAMPLES_DIR / "timbrature_compact"),
    ("sample_timbrature_elenco", SAMPLES_DIR / "timbrature_elenco"),
]

selected: list[tuple[str, Path]] = []
for employee, folder in sample_folders:
    pdfs = sorted(folder.glob("*.pdf"))[:MAX_PDFS_PER_FOLDER]
    selected.extend((employee, pdf) for pdf in pdfs)

if not selected:
    raise RuntimeError("No sample PDFs found.")

records: list[dict[str, object]] = []
for employee, pdf_path in selected:
    extracted = extract_best_local(pdf_path)

    out_dir = TEXT_DIR / employee
    out_dir.mkdir(parents=True, exist_ok=True)
    out_txt = out_dir / f"{pdf_path.stem}.txt"
    out_txt.write_text(extracted["text"], encoding="utf-8")

    records.append(
        {
            "employee": employee,
            "pdf": str(pdf_path.relative_to(REPO_ROOT)),
            "txt": str(out_txt.relative_to(REPO_ROOT)),
            "selected_mode": extracted["mode"],
            "tried_vertical": bool(extracted["tried_vertical"]),
            "normal_score": extracted["normal_score"],
            "vertical_score": extracted["vertical_score"],
            "text_chars": len(extracted["text"]),
        }
    )

extract_df = pd.DataFrame(records).sort_values(["employee", "pdf"])
display(extract_df)
print(f"Generated txt files: {len(extract_df)}")

## 2) Raw parsing unit checks (`raw_text_parsing`)

Spot-check parser primitives on real extracted lines.

In [ ]:
from src.raw_text_parsing import detect_doc_format, extract_events, line_has_event, parse_day_header, resolve_year_month

txt_files = sorted(TEXT_DIR.rglob("*.txt"))
sample_txt = first_or_none(txt_files)
if sample_txt is None:
    raise RuntimeError("No txt files found in notebook workspace.")

text = sample_txt.read_text(encoding="utf-8", errors="replace")
doc_format = detect_doc_format(text)
year, month = resolve_year_month(text, sample_txt)

print("sample_txt:", sample_txt.relative_to(REPO_ROOT))
print("doc_format:", doc_format)
print("resolved year/month:", year, month)

rows = []
for line in text.splitlines()[:200]:
    raw = line.strip()
    if not raw:
        continue
    header = parse_day_header(raw)
    has_event = line_has_event(raw)
    events = extract_events(raw)
    if header is not None or has_event:
        rows.append(
            {
                "raw": raw,
                "day_header": header,
                "has_event": has_event,
                "event_count": len(events),
                "events": [f"{e.kind} {e.time_str}" for e in events],
            }
        )

display(pd.DataFrame(rows).head(20))

## 3) Run `extract_days_from_text_raw`

In [ ]:
days_report_path = REPORTS_DIR / "extract_days_from_text_raw.report.json"

run_module(
    "src.extract_days_from_text_raw",
    "--input-dir", str(TEXT_DIR),
    "--out-dir", str(PARSED_DIR),
    "--out-name", "days.csv",
    "--report-json", str(days_report_path),
    "--verbose",
)

days_report = json.loads(days_report_path.read_text(encoding="utf-8"))
display(pd.DataFrame([days_report["stats"]]))

days_files = sorted(PARSED_DIR.rglob("days.csv"))
print("days.csv files:", len(days_files))
preview_csv(days_files[0])

## 4) Run `extract_events_from_days_raw`

In [ ]:
events_report_path = REPORTS_DIR / "extract_events_from_days_raw.report.json"

run_module(
    "src.extract_events_from_days_raw",
    "--input-dir", str(PARSED_DIR),
    "--days-name", "days.csv",
    "--out-name", "events_from_days_raw.csv",
    "--report-json", str(events_report_path),
    "--verbose",
)

events_report = json.loads(events_report_path.read_text(encoding="utf-8"))
display(pd.DataFrame([events_report["stats"]]))

events_files = sorted(PARSED_DIR.rglob("events_from_days_raw.csv"))
print("events_from_days_raw.csv files:", len(events_files))
preview_csv(events_files[0])

## 5) Run `filter_midnight_events_from_days_raw`

In [ ]:
clean_report_path = REPORTS_DIR / "events_from_days_raw.clean_midnight.report.json"
removed_csv_path = REPORTS_DIR / "events_from_days_raw.midnight_removed.csv"

run_module(
    "src.filter_midnight_events_from_days_raw",
    "--input-dir", str(PARSED_DIR),
    "--events-name", "events_from_days_raw.csv",
    "--out-name", "events_from_days_raw.cleaned.csv",
    "--report-json", str(clean_report_path),
    "--removed-csv", str(removed_csv_path),
    "--verbose",
)

clean_report = json.loads(clean_report_path.read_text(encoding="utf-8"))
display(pd.DataFrame([clean_report["stats"]]))

cleaned_files = sorted(PARSED_DIR.rglob("events_from_days_raw.cleaned.csv"))
print("events_from_days_raw.cleaned.csv files:", len(cleaned_files))
preview_csv(cleaned_files[0])

if removed_csv_path.exists():
    print("Removed rows preview:")
    display(pd.read_csv(removed_csv_path).head(10))

## 6) Run `pair_employee_events_from_days_raw`

In [ ]:
pairs_report_path = REPORTS_DIR / "pair_employee_events_from_days_raw.report.json"

run_module(
    "src.pair_employee_events_from_days_raw",
    "--input-dir", str(PARSED_DIR),
    "--output-dir", str(PAIRS_DIR),
    "--events-name", "events_from_days_raw.cleaned.csv",
    "--report-json", str(pairs_report_path),
    "--max-gap-hours", "16",
    "--verbose",
)

pairs_report = json.loads(pairs_report_path.read_text(encoding="utf-8"))
display(pd.DataFrame([pairs_report["stats"]]))

pairs_files = sorted(PAIRS_DIR.glob("*.pairs.csv"))
print("pairs files:", len(pairs_files))
if pairs_files:
    preview_csv(pairs_files[0])

## 7) Run `turni_enrichment`

In [ ]:
enrichment_stats_path = REPORTS_DIR / "turni_enrichment.stats.json"

run_module(
    "src.turni_enrichment",
    "--input-dir", str(PAIRS_DIR),
    "--out-dir", str(ENRICHED_DIR),
    "--min-hours", "6",
    "--stats-json", str(enrichment_stats_path),
    "--verbose",
)

enrichment_stats = json.loads(enrichment_stats_path.read_text(encoding="utf-8"))
display(pd.DataFrame([enrichment_stats["stats"]]))

enriched_files = sorted(ENRICHED_DIR.glob("*.enriched.csv"))
print("enriched files:", len(enriched_files))
if enriched_files:
    preview_csv(enriched_files[0])

## 8) Run `turni_employee_summary`

In [ ]:
summary_csv_path = AGG_DIR / "turni_employee_summary.csv"
summary_json_path = AGG_DIR / "turni_employee_summary.json"

run_module(
    "src.turni_employee_summary",
    "--enriched-dir", str(ENRICHED_DIR),
    "--out", str(summary_csv_path),
    "--year-start", "2014",
    "--year-end", "2026",
    "--format", "csv",
    "--verbose",
)

run_module(
    "src.turni_employee_summary",
    "--enriched-dir", str(ENRICHED_DIR),
    "--out", str(summary_json_path),
    "--year-start", "2014",
    "--year-end", "2026",
    "--format", "json",
    "--verbose",
)

preview_csv(summary_csv_path)

summary_json = json.loads(summary_json_path.read_text(encoding="utf-8"))
print("summary json stats:")
display(pd.DataFrame([summary_json["stats"]]))

## 9) Quick artifact inventory

In [ ]:
for label, folder, pattern in [
    ("TXT", TEXT_DIR, "*.txt"),
    ("DAYS", PARSED_DIR, "days.csv"),
    ("EVENTS RAW", PARSED_DIR, "events_from_days_raw.csv"),
    ("EVENTS CLEAN", PARSED_DIR, "events_from_days_raw.cleaned.csv"),
    ("PAIRS", PAIRS_DIR, "*.pairs.csv"),
    ("ENRICHED", ENRICHED_DIR, "*.enriched.csv"),
]:
    files = sorted(folder.rglob(pattern))
    print(f"{label:12} -> {len(files)}")
    for path in files[:3]:
        print("   ", path.relative_to(REPO_ROOT))

print("\nReports:")
for report in sorted(REPORTS_DIR.rglob("*.json")):
    print("  ", report.relative_to(REPO_ROOT))